In [0]:
create or replace temp view vw_sales_silver_clean as (
  with cleaned as (
    select
       cast(id_nf                      as string)            as invoice_id
      ,cast(data_venda                 as date)              as sale_date
      ,cast(cast(cliente_id            as double) as bigint) as customer_id
      ,cast(cliente_nome               as string)            as customer_name
      ,cast(marca_carro                as string)            as car_brand
      ,cast(modelo_carro               as string)            as car_model
      ,cast(produto_peca               as string)            as car_part
      ,zeroifnull(cast(valor_unitario  as double))           as part_unit_price
      ,zeroifnull(cast(cast(quantidade as double) as int))   as part_quantity
      ,zeroifnull(cast(custo_unitario  as double))           as part_unit_cost
      ,cast(cast(loja_id               as double) as int)    as store_id
      ,cast(loja_nome                  as string)            as store_name
      ,cast(case
        when grupo_loja = 'Grupo Atlântic'                   then 'Grupo Atlantico'
        when grupo_loja = 'Grupo Atlantico'                  then 'Grupo Atlantico'
        when grupo_loja = 'Grupo Pampa'                      then 'Grupo Pampas'
        when grupo_loja = 'Grupo Pampas'                     then 'Grupo Pampas'
        when grupo_loja = 'Grupo Serras'                     then 'Grupo Serra'
        when grupo_loja = 'Grupo Serra'                      then 'Grupo Serra'
        when grupo_loja = 'Grupo Tytan'                      then 'Grupo Titan'
        when grupo_loja = 'Grupo Titan'                      then 'Grupo Titan'
        else null
       end                             as string)            as store_group
      ,cast(vendedor_id                as string)            as seller_id
      ,cast(vendedor_nome              as string)            as seller_name
      ,ingestion_timestamp                                   as ingestion_timestamp
      ,source_file                                           as source_file
    from workspace.pj_sales.tb_sales_bronze
    where zeroifnull(cast(cast(quantidade as double) as int))     >= 0 -- Os valores negativos ou iguais a 0 é necessário serem tirados?
      and zeroifnull(cast(custo_unitario  as double))             >= 0 -- Custos e valores menores do que 0 são alguma regra de negócio?
      and zeroifnull(cast(valor_unitario  as double))             >= 0 -- **
  )
  select
       trim(upper(invoice_id))                               as invoice_id -- Há diferença de id em tamanho da string?
      ,date_format(sale_date, 'yyyy-MM-dd')                  as sale_date
      ,customer_id                                           as customer_id
      ,trim(upper(customer_name))                            as customer_name
      ,trim(upper(car_brand))                                as car_brand
      ,trim(upper(car_model))                                as car_model
      ,trim(upper(car_part))                                 as car_part
      ,part_unit_price                                       as part_unit_price
      ,part_quantity                                         as part_quantity
      ,part_unit_cost                                        as part_unit_cost
      ,store_id                                              as store_id
      ,trim(upper(store_name))                               as store_name
      ,trim(upper(store_group))                              as store_group
      ,trim(upper(seller_id))                                as seller_id -- Há diferença de id em tamanho da string?
      ,trim(upper(seller_name))                              as seller_name
      ,ingestion_timestamp                                   as ingestion_timestamp
      ,source_file                                           as source_file
  from cleaned
)

In [0]:
create or replace temp view vw_sales_silver_dedup as (
  with dedup as (
    select
       invoice_id
      ,sale_date
      ,customer_id
      ,customer_name
      ,car_brand
      ,car_model
      ,car_part
      ,part_unit_price
      ,part_quantity
      ,part_unit_cost
      ,store_id
      ,store_name
      ,store_group
      ,seller_id
      ,seller_name
      ,ingestion_timestamp
      ,source_file
      ,row_number() over 
        (partition by invoice_id 
          order by 
             sale_date             desc 
            ,ingestion_timestamp   desc
        )                          as order_by_date --! Resolvido no MERGE, chance de atualização e perda de dados
    from vw_sales_silver_clean
  )
  select
     invoice_id
    ,sale_date
    ,customer_id
    ,customer_name
    ,car_brand
    ,car_model
    ,car_part
    ,part_quantity
    ,part_unit_price
    ,part_unit_cost
    --,round(part_unit_price * part_quantity, 2)                    as part_total_price
    --,round(part_unit_cost * part_quantity, 2)                     as part_total_cost
    --,round(part_total_price - part_total_cost, 2)                 as part_profit -- ! Removido e realocado na gold
    ,store_id
    ,store_name
    ,store_group
    ,seller_id
    ,seller_name
    ,ingestion_timestamp
    ,from_utc_timestamp(current_timestamp(), 'America/Sao_Paulo')   as updating_timestamp
    ,source_file
  from dedup
  where order_by_date = 1
)


In [0]:
create table if not exists workspace.pj_sales.tb_sales_silver
using delta
as
select
  *
from vw_sales_silver_dedup

In [0]:
merge into workspace.pj_sales.tb_sales_silver tg -- target
using vw_sales_silver_dedup src -- source
on tg.invoice_id = src.invoice_id
when matched then
  update set
    --s.invoice_id            = sd.invoice_id --! Não precisa pois é o mesmo, chave
     tg.sale_date             = coalesce(src.sale_date,         tg.sale_date) -- coalesce
    ,tg.customer_id           = coalesce(src.customer_id,       tg.customer_id)
    ,tg.customer_name         = coalesce(src.customer_name,     tg.customer_name)
    ,tg.car_brand             = coalesce(src.car_brand,         tg.car_brand)
    ,tg.car_model             = coalesce(src.car_model,         tg.car_model)
    ,tg.car_part              = coalesce(src.car_part,          tg.car_part)
    ,tg.part_unit_price       = coalesce(src.part_unit_price,   tg.part_unit_price)
    ,tg.part_quantity         = coalesce(src.part_quantity,     tg.part_quantity)
    ,tg.part_unit_cost        = coalesce(src.part_unit_cost,    tg.part_unit_cost)
    --,tg.part_total_price    = coalesce(src.part_total_price,  tg.part_total_price)
    --,tg.part_total_cost     = coalesce(src.part_total_cost,   tg.part_total_cost)
    --,tg.part_profit         = coalesce(src.part_profit,       tg.part_profit)
    ,tg.store_id              = coalesce(src.store_id,          tg.store_id)
    ,tg.store_name            = coalesce(src.store_name,        tg.store_name)
    ,tg.store_group           = coalesce(src.store_group,       tg.store_group)
    ,tg.seller_id             = coalesce(src.seller_id,         tg.seller_id)
    ,tg.seller_name           = coalesce(src.seller_name,       tg.seller_name)
    ,tg.updating_timestamp    = from_utc_timestamp(current_timestamp(), 'America/Sao_Paulo')
    ,tg.source_file           = coalesce(src.source_file,       tg.source_file) -- Atualização de dados veio de um novo arquivo, arquivo antigo passa a ser desconsiderado?
when not matched then
  insert *

In [0]:
drop view vw_sales_silver_clean;
drop view vw_sales_silver_dedup;